# အခန်း ၁၆ - Microsoft Foundry ဖြင့် စမတ်အေဂျင့်များကို တိုးချဲ့စွာ တပ်ဆင်ခြင်း

ဤ notebook တွင် သင်သည် အတုလုပ်ထားသော ကုမ္ပဏီ Contoso အတွက် **ထုတ်လုပ်မှုအသင့်ရှိသည့် ဖောက်သည်ဝန်ဆောင်မှု အေဂျင့်တစ်ခု** ကို တည်ဆောက်မည်ဖြစ်သည်။ ယခင်သင်ခန်းစာများနှင့် မတူဘဲ၊ ဒီမှာ အဓိက အချက်မှာ အေဂျင့်၏ အတွေးအခေါ် လုပ်ငန်းစဉ်မဟုတ်ပါဘူး — ဒါဟာ စီမံခန့်ခွဲမှုအဆင့်သေးသေးလေးဖြစ်ပေမယ့် အရေးကြီးတဲ့ အစိတ်အပိုင်းများကို အကြောင်းအရာအတွင်းမှာ ပတ်ဝန်းကျင်အဖြစ် ထည့်သွင်းထားပါတယ်။

၁။ **ကိရိယာခေါ်ဆိုမှု** — အမှာစာကြည့်ရှုခြင်းနှင့် လက်မှတ်ဖန်တီးခြင်း။
၂။ **RAG** — သတင်းအချက်အလက်အခြေခံမှ မူဝါဒဖြေကြားချက်များ။
၃။ **မှတ်ဉာဏ်** — ဖောက်သည်ကို ဆက်တိုက်နားလည်သ 기억ရန်။
၄။ **ပုံစံလမ်းကြောင်းပြန်ကြားခြင်း** — ရိုးရှင်းသော တောင်းဆိုမှုများကို သေးငယ်သော မော်ဒယ်ထံ ပို့ခြင်း၊ ခက်ခဲသော တောင်းဆိုမှုများကို ကြီးမားသော မော်ဒယ်ထံ ပို့ခြင်း။
၅။ **တုံ့ပြန်ချက် ကက်ရှ်** — မော်ဒယ်ခေါ်စဉ်မရှိဘဲ ပြန်ပါတယ်သော မေးခွန်းများကို ဖြေကြားခြင်း။
၆။ **လူ့အတည်ပြုချက်** — ခွင့်ပြန်ငွေပြန်လုပ်မှုများသည် သတ်မှတ်ထားသည့် အဆင့်ထက် မကျော်လွန်ပါက ခြိမ်းခြောက်မှုရပ်ဆိုင်းမှု။
၇။ **အကဲဖြတ်ခြင်း ဝင်ခွင့်** — ဆိုးရွားသော ထုတ်လုပ်မှုတစ်ခုကို တားဆီးရန် အော့ဖ်လိုင်းစမ်းသပ်မှုဆိုင်ရာ စနစ်။
၈။ **ကြည့်ရှုနိုင်မှု** — OpenTelemetry သုံး၍ တောင်းဆိုမှုတိုင်းတွင် ချဲ့ထွင်ခြင်း။

အပိုင်းတိုင်းသည် ကိုယ်ပိုင်အသီးသီးဖြစ်ပြီး รันနိုင်သည်။ စာကြောင်းတိုင်းကို ဖတ်ပါ — ထုတ်လုပ်မှု၏ အခြေခံအစိတ်အပိုင်းများကို ဂရုစိုက်၍ သေးငယ်သွားအောင် ထိန်းသိမ်းထားသည်။


## စတင်ခြင်း

ဤ notebook ကို လုပ်ဆောင်ရန်မတိုင်မီ၊ သင်မှာရှိရမည့်အရာများမှာ-

1. **Microsoft Foundry project** တစ်ခု၊ chat model တစ်ခု (ဥပမာ `gpt-5-mini`) ဖြန့်ထားပြီးဖြစ်ရမည်။
2. **Azure CLI ဖြင့် စာရင်းဝင်ပြီးဖြစ်ရမည်** — သင်၏ terminal တွင် `az login` ကို အလုပ်လုပ်ပါ။
3. **လိုအပ်သော အပြင်ဘက်ပတ်ဝန်းကျင်အရင်းအမြစ်များကို သတ်မှတ်ထားရမည်။**
   - `AZURE_AI_PROJECT_ENDPOINT` — မိမိ Microsoft Foundry project ၏ endpoint။
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — မိမိ ဖြန့်ထားသော model ၏ အမည်။

RAG အပိုင်းတွင် `AZURE_SEARCH_SERVICE_ENDPOINT` နှင့် `AZURE_SEARCH_API_KEY` တို့ သတ်မှတ်ထားပါက **Azure AI Search** ကို အသုံးပြုကာ၊ မရှိပါကအတွင်းမှ ရှာဖွေမှုကို အသုံးပြု၍ notebook ကို Search resource မရှိပဲလည်း လည်ပတ်နိုင်စေပါသည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. ကိရိယာများ

ထုတ်လုပ်မှုပစ္စည်းများသည် အမှန်တကယ် စနစ်များနှင့် တိုက်ရိုက်အလုပ်လုပ်ကြသည်။ ဒီမှာ ကျွန်ုပ်တို့ plain Python လုပ်ဆောင်ချက်များဖြင့် အမိန့်ဒေတာဘေ့စ်နှင့် တစ်စိတ်တစ်ပိုင်း တကယ်ရှိတဲ့ စက်တင်စနစ်တစ်ခုကို အတုလုပ်ထားသည်။ `@tool` decorator သည် ၎င်းတို့ကို agent ထံ ဖြန့်ဝေသည်။

`issue_refund` သည် အစားနည်းအလွန်အမင်းသော အကြိမ်စားအတွက် `approval_mode="always_require"` ကို အသုံးပြုသည်။ ၎င်းသည် နောက်ပိုင်းတွင် အသုံးပြုမည့် လူနှင့် နှောင့်နှေးခြင်းသော primitive ဖြစ်သည်။


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — မူဝါဒ အသိပညာ အခြေခံ

မူဝါဒဆိုင်ရာ မေးခွန်းများ ("သင့်ပြန်လည်ပေးပို့ချပ်ကာလ ဘယ်လောက်လဲ?") ကို မော်ဒယ်ရဲ့မှတ်ဥာဏ်မှ မဟုတ်ဘဲ တရားဝင်အရင်းအမြစ်မှ ဖြေကြားရမည်။ ကျွန်ုပ်တို့သည် အသိပညာအခြေခံစာအုပ်အသေးကြီးကို ရှာဖွေရေးကိရိယာအဖြစ် ထည့်သွင်းထားသည်။

ထုတ်လုပ်မှုတွင် ဤသည်မှာ **Azure AI Search** ဖြစ်ပြီး၊ ဤနေရာတွင် နုတ်ဘွတ်ဘုတ်ကို မည်သည့်နေရာတွင်မဆို ပြေးနိုင်ရန်အတွက် စကားဝှက် ရှာဖွေရေးလုပ်ဆောင်ချက်ကို မှတ်ဥာဏ်အတွင်း ပံ့ပိုးပေးထားပြီး ပတ်ဝန်းကျင်အပြောင်းအလဲအတိုင်း Azure AI Search သို့ အလိုအလျောက်ဆွဲပြောင်းသည်။


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. မှတ်ဥာဏ်

ဘယ်သူနဲ့ပြောနေတာမေ့နေတဲ့ အထောက်အပံ့ကိုယ်စားလှယ်က မကောင်းတဲ့ အထောက်အပံ့ကိုယ်စားလှယ်ပါ။ ကျွန်တော်တို့ဟာ ဖောက်သည်တဦးချင်းစီအတွက် သေးငယ်တဲ့ ပရိုဖိုင် စတော့မျိုး တစ်ခုထားရှိပြီး အဲ့ဒီကို အက်ဂျင့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့့်။ ထုတ်လုပ်မှုမှာ ဟာမှတ်ဥာဏ်ဝန်ဆောင်မှုတစ်ခုဖြစ်ပါတယ် (သင်ခန်းစာ ၁၃ ကိုကြည့်ပါ)။ ဒီမှာတော့ dict တစ်ခုက ပုံစံကို ပေါ်လွင်အောင်လုပ်ပေးပါတယ်။


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## ၄ နှင့် ၅။ မော်ဒယ်လမ်းညွှန်ခြင်းနှင့် တုံ့ပြန်မှုသိုလှောင်ခြင်း

တစ်ခုတည်းသော မေးခွန်းကိုင်တွယ်သူတွင် ဆက်သွယ်ထားသော စရိတ်လျော့ချမှုနှစ်ခု -

- **လမ်းညွှန်မှု**: စျေးနူန်းစရိတ်နည်းသော heuristic classifier တစ်ခုက မေးခွန်းသည် မော်ဒယ်သေး သို့မဟုတ် မော်ဒယ်ကြီး မလိုအပ်ပါကဆုံးဖြတ်ပေးသည်။
- **သိုလှောင်မှု**: ပုံမှန် ပြန်လည်မေးမြန်းထားသော မေးခွန်းများကို မော်ဒယ်ခေါ်ဆိုမှုမရှိဘဲ cache မှတိုက်ရိုက်ပေးဆောင်သည်။

ဒီ့မှာ အဆိုပါ classifier ကို ရိုးရှင်းစွာ ရေးသားထားသည်။ ထုတ်လုပ်မှုတွင် traffic ဆန့်အတည်ပြုပြီး Foundry ၏ Model Router ဖြင့် အစားထိုးနိုင်သည်။


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## ၆ & ၈။ အေးဂျင့်၊ လူ့အတည်ပြုမှုနှင့် ကြည့်ရှုနိုင်မှု

ယခုမှာ အထက်ပါကိရိယာများမှ အေးဂျင့်ကို စုစည်းပြီး ကူးယူထားသည့် တောင်းဆိုမှု တစ်ခုချင်းစီကို OpenTelemetry span ဖြစ်စေပြီး လုပ်ဆောင်သည်။ `handle_support_request` လုပ်ဆောင်ချက်သည် ထုတ်လုပ်မှု တောင်းဆိုမှု ကိုင်တွယ်သူဖြစ်သည်။ cache → route → trace → run → cache ဖြစ်သည်။

လူ့အတည်ပြုမှုကို ကိုယ့်ဟာစနစ်က ကိုင်တွယ်ပေးသည်။ `issue_refund` မှာ `approval_mode="always_require"` ဖြစ်သောကြောင့် run သည် မိနစ်တစ်ခုနားပြီး လူ့အတည်ပြုမှု တောင်းဆိုချက်ကို ပေါ်လာအောင် ပြသပေးပြီး ကျွန်ုပ်တို့က အတည်ပြုပြီး ဖြေရှင်းသွားသည်။


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## ၇။ အဆင့်သတ်မှတ်ခြင်းတံခါး

ဒါက သင်ခန်းစာ၏ ဖြတ်တောက်ပေးရာ တံခါးဖြစ်ပြီး၊ agent ကို ပေးသည့် offline စမ်းသပ်ချက် များမှတ်တမ်းသည် agent ကို အကဲဖြတ်ပေးပြီး၊ ဖြတ်တောက်ရာ အဆင့်မှတ်တမ်းသည် ကန့်သတ်ချက်တစ်ခုကျော်လွန်မှသာ deployment တက်နိုင်သည်။ ဒီမှာ scorer သည် notebook ကို ကိုယ်ပိုင်အတွင်းမှာ ထိန်းသိမ်းရန် အတွက် ရိုးရှင်းသော keyword-overlap စစ်ဆေးမှု ဖြစ်သည်။ ထုတ်လုပ်ရေးတွင် သင်သည် LLM တစ်ခုကို တရားစဉ်သူအဖြစ် သုံးမည် ဖြစ်သလို သို့မဟုတ် framework evaluator (သင်ခန်းစာ ၁၀ ကို ကြည့်ပါ) ကို အသုံးပြုရမည်ဖြစ်သည်။


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ပေါင်းစပ်ခြင်း - စမ်းသပ်ထုတ်ဝေမည်

အောက်ပါဆဲလ်တွင် သင်ခန်းစာမှဖေါ်ပြသည့် အကြောင်းအရာ အားလုံးပါရှိသည် - သုံးသပ်တံခါးကို ပြေးဆွဲပါ၊ ဖြတ်သန်းပါက "တင်သွင်း" ပါ။ ဒါဟာ သင် CI ထဲမှာ ဖောင်ဒရီအေဂျင့်ဝန်ဆောင်မှုသို့ အေဂျင့်ဗားရှင်းတစ်ခုကို တင်ပေးမယ့် နမူနာပုံစံဖြစ်ပါတယ်။


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## အကျဉ်းချုပ်

သင်သည် အ operational ကိစ္စရပ်များအားလုံးအား ထည့်သွင်းထားသော ထုတ်လုပ်မှုခေတ်ပြိုင် ဖောက်သည်ထောက်ခံရေးအေးဂျင့်တစ်ယောက်ကို စုပေါင်းထားပါသည်။

- **ကိရိယာများ၊ RAG နှင့် မှတ်ဉာဏ်** က အေးဂျင့်အား စွမ်းရည်နှင့် အကြောင်းအရာ ပေးသည်။
- **မော်ဒယ်လ်လမ်းညွှန်ခြင်းနှင့် သိုလှောင်ခြင်း** က အချိန်နှင့် ကျသင့်ငွေကို ထိန်းချုပ်ထားသည်။
- **လူ့အသိအမှတ်ပြုခြင်း** က စိတ်လှုပ်ရှားစရာကြီးသော လုပ်ဆောင်ချက်များ (သန့်ရှင်းရေးကြီးမှုများကဲ့သို့) ကို ကာကွယ်သည်။
- **သုံးသပ်မှုတံခါး** က မကောင်းသော ထွက်ပေါက်များကို ပို့ခွင့်မပြုမီ တားဆီးသည်။
- **စောင့်ကြည့်ခြင်း** က မည်သည့်တောင်းဆိုချက်မဆို တွေ့မြင်နိုင်စေသည်။

### စိန်ခေါ်မှု

ဤအေးဂျင့်ကို တိုးချဲ့ပါ -

1. **မော်ဒယ်များ များစွာကို ထောက်ပံ့ခြင်း** — တတိယ "ရည်မှန်းချက်" အဆင့်တစ်ခု ထပ်ထည့်ပြီး တောင်းဆိုမှုများ/ပူပန်မှုများကို အဲဒီမှာ လမ်းညွှန်ပါ။
2. **သုံးသပ်မှုတံခါးများ ထည့်ခြင်း** — `TEST_CASES` ကို အာမခံဖြန့်ချိမှုအခြေအနေများဖြင့် တိုးချဲ့ပြီး တံခါးက လူလွတ်မှုနောက်ပြန်ကြောတားဆီးမှုကို အတည်ပြုပါ။
3. **ကုန်ကျစရိတ် သတိပြု လမ်းညွှန်ခြင်း** — တောင်းဆိုမှုတစ်ချက်စီအတွက် ခန့်မှန်းကုန်ကျစရိတ် (သေးငယ်မှု၊ ကြီးမားမှု၊ သိုလှောင်မှု) ကို လိုက်လံထားပြီး ကုန်ကျစရိတ်အစီရင်ခံစာကို အမြှုပ်မေးခွန်းတွဲပြီး ပြပါ။

နောက်တစ်ခေါက်သင်ခန်းစာတွင် သငျသညျဆနျ့မညျကမ္ဘာပေါ်ရှိ သင်၏စက်ပေါ်တွင် Microsoft Foundry Local နှင့် Qwen ဖြင့် အေးဂျင့်ကို လုံးဝကိုယ်ပိုင်ပုံစံဖြင့် ဖြတ်သန်းသွားမည် ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
